# Yelp — EmbeddingANN

**Goal**: Train an ANN that learns dense embedding representations for
all 5 categorical features, combined with the 6 numerical features.

**Categorical embeddings:**
| Feature | Type | Embedding strategy |
|---------|------|-------------------|
| `state` | 14 unique | Direct lookup |
| `city` | 758 unique | Direct lookup |
| `categories` | 1,166 tokens | Mean-pool (multi-label) |
| `user_id` | High cardinality | Lookup (≥5 reviews filter) |
| `business_id` | High cardinality | Lookup (≥5 reviews filter) |

**Reads from**: `data/train_features.csv`
**Saves to**  : `embeddings/`


## Roadmap
```
STEP 1  — Imports & paths
STEP 2  — Load data
STEP 3  — Label encode all categoricals
STEP 4  — Scale numerical features
STEP 5  — PyTorch Dataset & DataLoader
STEP 6  — Model definition (EmbeddingANN)
STEP 7  — Train with early stopping
STEP 8  — Evaluate
STEP 9  — Save embeddings to disk
```

---
## Step 1 — Imports & Paths

In [ ]:
import os, json, time, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data      import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics       import mean_squared_error, mean_absolute_error

SEED   = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"Device: {DEVICE}")


In [ ]:
DATA_DIR = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\data"
EMB_DIR  = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\embeddings"
os.makedirs(EMB_DIR, exist_ok=True)
print(f"Data dir : {DATA_DIR}")
print(f"Emb dir  : {EMB_DIR}")


---
## Step 2 — Load Data

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train_features.csv'))
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val_features.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test_features.csv'))
print(f"Train : {df_train.shape}  Val : {df_val.shape}  Test : {df_test.shape}")


In [ ]:
NUM_COLS  = [
    'user_avg_rating',    'user_rating_count',
    'business_avg_rating','business_rating_count', 'business_rating_std',
    'days_since_review'
]
TARGET    = 'stars'
CAT_STATE = 'state'
CAT_CITY  = 'city'
CAT_CATS  = 'categories'


---
## Step 3 — Label Encode All Categoricals

Five separate encoders — one per categorical feature.
All vocabularies built from train only.
Unseen values in val/test → index 0 (<UNK>).

user_id and business_id: filtered to ≥5 reviews to prevent
embedding table memorisation (same fix as Amazon).


In [ ]:
MIN_REVIEWS = 5

# ── State encoder (14 unique — no filtering needed) ──
state_vocab = ['<UNK>'] + sorted(df_train[CAT_STATE].dropna().unique().tolist())
state2idx   = {s: i for i, s in enumerate(state_vocab)}

# ── City encoder (758 unique — no filtering needed) ──
city_vocab = ['<UNK>'] + sorted(df_train[CAT_CITY].dropna().unique().tolist())
city2idx   = {c: i for i, c in enumerate(city_vocab)}

# ── Category token encoder (mean-pool approach) ──
all_tokens = set()
for c in df_train[CAT_CATS].dropna():
    all_tokens.update(c.split('|'))
all_tokens.discard('<UNK>')
cat_vocab = ['<UNK>'] + sorted(all_tokens)
cat2idx   = {c: i for i, c in enumerate(cat_vocab)}

# ── User encoder (≥5 reviews filter) ──
user_counts  = df_train['user_id'].value_counts()
active_users = set(user_counts[user_counts >= MIN_REVIEWS].index)
user_vocab   = ['<UNK>'] + sorted(active_users)
user2idx     = {u: i for i, u in enumerate(user_vocab)}

# ── Business encoder (≥5 reviews filter) ──
biz_counts      = df_train['business_id'].value_counts()
active_biz      = set(biz_counts[biz_counts >= MIN_REVIEWS].index)
biz_vocab       = ['<UNK>'] + sorted(active_biz)
biz2idx         = {b: i for i, b in enumerate(biz_vocab)}

n_states = len(state2idx); n_cities  = len(city2idx)
n_cats   = len(cat2idx);   n_users   = len(user2idx)
n_biz    = len(biz2idx)

print(f"State vocab    : {n_states}")
print(f"City vocab     : {n_cities}")
print(f"Category vocab : {n_cats}")
print(f"User vocab     : {n_users:,}  (active ≥{MIN_REVIEWS} reviews)")
print(f"Business vocab : {n_biz:,}  (active ≥{MIN_REVIEWS} reviews)")


In [ ]:
MAX_CATS = 10  # max category tokens per business (truncate/pad)

def encode_split(df):
    """Encode all 5 categoricals for a dataframe split."""
    state_idx = df[CAT_STATE].fillna('<UNK>').map(
        lambda x: state2idx.get(x, 0)).values.astype(np.int64)
    city_idx  = df[CAT_CITY].fillna('<UNK>').map(
        lambda x: city2idx.get(x, 0)).values.astype(np.int64)

    # Categories: padded sequence of token indices (-1 = padding)
    cat_seqs = []
    for c_str in df[CAT_CATS].fillna('<UNK>'):
        tokens  = c_str.split('|')[:MAX_CATS]
        indices = [cat2idx.get(t, 0) for t in tokens]
        indices += [-1] * (MAX_CATS - len(indices))
        cat_seqs.append(indices)
    cat_idx = np.array(cat_seqs, dtype=np.int64)

    user_idx = df['user_id'].map(
        lambda x: user2idx.get(x, 0)).values.astype(np.int64)
    biz_idx  = df['business_id'].map(
        lambda x: biz2idx.get(x, 0)).values.astype(np.int64)

    return state_idx, city_idx, cat_idx, user_idx, biz_idx

tr_state, tr_city, tr_cats, tr_user, tr_biz = encode_split(df_train)
v_state,  v_city,  v_cats,  v_user,  v_biz  = encode_split(df_val)
te_state, te_city, te_cats, te_user, te_biz  = encode_split(df_test)

# Validate all indices
assert tr_state.max() < n_states and tr_city.max() < n_cities
valid = tr_cats[tr_cats >= 0]
assert valid.max() < n_cats
assert tr_user.max() < n_users and tr_biz.max() < n_biz
print("All indices valid ✓")


In [ ]:
# Save all encoders for classical model notebooks
encoders = {
    'state_encoder'   : state2idx,
    'city_encoder'    : city2idx,
    'cat_encoder'     : cat2idx,
    'user_encoder'    : user2idx,
    'business_encoder': biz2idx,
}
for fname, enc in encoders.items():
    with open(os.path.join(EMB_DIR, f'{fname}.json'), 'w') as f:
        json.dump(enc, f)
print("Encoders saved.")


---
## Step 4 — Scale Numerical Features

In [ ]:
scaler      = StandardScaler()
X_train_num = scaler.fit_transform(df_train[NUM_COLS]).astype(np.float32)
X_val_num   = scaler.transform(df_val[NUM_COLS]).astype(np.float32)
X_test_num  = scaler.transform(df_test[NUM_COLS]).astype(np.float32)

y_train = df_train[TARGET].values.astype(np.float32)
y_val   = df_val[TARGET].values.astype(np.float32)
y_test  = df_test[TARGET].values.astype(np.float32)
print(f"Numerical matrix: {X_train_num.shape}")


---
## Step 5 — Dataset & DataLoader

In [ ]:
class YelpDataset(Dataset):
    def __init__(self, num, state, city, cats, user, biz, labels):
        self.num   = torch.tensor(num,   dtype=torch.float32)
        self.state = torch.tensor(state, dtype=torch.long)
        self.city  = torch.tensor(city,  dtype=torch.long)
        self.cats  = torch.tensor(cats,  dtype=torch.long)
        self.user  = torch.tensor(user,  dtype=torch.long)
        self.biz   = torch.tensor(biz,   dtype=torch.long)
        self.y     = torch.tensor(labels,dtype=torch.float32)

    def __len__(self): return len(self.y)

    def __getitem__(self, idx):
        return (self.num[idx], self.state[idx], self.city[idx],
                self.cats[idx], self.user[idx], self.biz[idx], self.y[idx])

def make_loaders(batch_size=512):
    tr = DataLoader(YelpDataset(X_train_num,tr_state,tr_city,tr_cats,tr_user,tr_biz,y_train),
                    batch_size=batch_size, shuffle=True)
    v  = DataLoader(YelpDataset(X_val_num,  v_state, v_city, v_cats, v_user, v_biz, y_val),
                    batch_size=batch_size, shuffle=False)
    te = DataLoader(YelpDataset(X_test_num, te_state,te_city,te_cats,te_user,te_biz,y_test),
                    batch_size=batch_size, shuffle=False)
    return tr, v, te

train_loader, val_loader, test_loader = make_loaders()
b = next(iter(train_loader))
print(f"Batch shapes — num:{b[0].shape} state:{b[1].shape} city:{b[2].shape} "
      f"cats:{b[3].shape} user:{b[4].shape} biz:{b[5].shape}")


---
## Step 6 — Model Definition (EmbeddingANN)

```
state      → Embedding(n_states, 4)   → direct lookup
city       → Embedding(n_cities, 8)   → direct lookup
categories → Embedding(n_cats,   8)   → mean-pool (masked)
user_id    → Embedding(n_users,  16)  → direct lookup + dropout
business_id→ Embedding(n_biz,    16)  → direct lookup + dropout
6 numerical (scaled)
                ↓
    concat [4+8+8+16+16+6 = 58]
    Linear(58→64) → ReLU → Dropout
    Linear(64→32) → ReLU → Dropout
    Linear(32→1)
```

Dropout on user/business embeddings (high cardinality → more overfitting risk).
State and city are low/medium cardinality — no dropout needed.


In [ ]:
class YelpEmbeddingANN(nn.Module):
    def __init__(self, n_states, n_cities, n_cats, n_users, n_biz, n_num,
                 state_dim=4, city_dim=8, cat_dim=8,
                 user_dim=16, biz_dim=16,
                 id_dropout=0.4, mlp_dropout=0.3):
        super().__init__()

        self.state_emb = nn.Embedding(n_states, state_dim, padding_idx=0)
        self.city_emb  = nn.Embedding(n_cities, city_dim,  padding_idx=0)
        self.cat_emb   = nn.Embedding(n_cats,   cat_dim,   padding_idx=0)
        self.user_emb  = nn.Embedding(n_users,  user_dim,  padding_idx=0)
        self.biz_emb   = nn.Embedding(n_biz,    biz_dim,   padding_idx=0)

        self.id_dropout  = nn.Dropout(id_dropout)

        # Store vocab sizes for clamping
        self.n_states = n_states; self.n_cities = n_cities
        self.n_cats   = n_cats;   self.n_users  = n_users
        self.n_biz    = n_biz

        mlp_in = state_dim + city_dim + cat_dim + user_dim + biz_dim + n_num
        self.mlp = nn.Sequential(
            nn.Linear(mlp_in, 64), nn.ReLU(), nn.Dropout(mlp_dropout),
            nn.Linear(64, 32),     nn.ReLU(), nn.Dropout(mlp_dropout),
            nn.Linear(32, 1)
        )

    def forward(self, num, state, city, cats, user, biz):
        # Clamp all indices
        state = state.clamp(0, self.n_states - 1)
        city  = city.clamp(0,  self.n_cities - 1)
        user  = user.clamp(0,  self.n_users  - 1)
        biz   = biz.clamp(0,   self.n_biz    - 1)

        # Categories: mean-pool with padding mask
        mask     = (cats >= 0).float()
        safe_cats= cats.clamp(0, self.n_cats - 1)
        cat_vecs = self.cat_emb(safe_cats) * mask.unsqueeze(-1)
        cat_vec  = cat_vecs.sum(1) / mask.sum(1).clamp(min=1).unsqueeze(-1)

        state_vec = self.state_emb(state)
        city_vec  = self.city_emb(city)
        user_vec  = self.id_dropout(self.user_emb(user))
        biz_vec   = self.id_dropout(self.biz_emb(biz))

        x = torch.cat([state_vec, city_vec, cat_vec, user_vec, biz_vec, num], dim=1)
        return self.mlp(x).squeeze(1)


model = YelpEmbeddingANN(n_states, n_cities, n_cats, n_users, n_biz,
                          len(NUM_COLS)).to(DEVICE)
total = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal parameters: {total:,}")


---
## Step 7 — Train with Early Stopping

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    model.train() if optimizer else model.eval()
    preds_all, labels_all = [], []
    ctx = torch.enable_grad() if optimizer else torch.no_grad()
    with ctx:
        for num_b,st_b,ci_b,ca_b,us_b,bz_b,y_b in loader:
            num_b,st_b,ci_b,ca_b,us_b,bz_b,y_b = (
                num_b.to(DEVICE),st_b.to(DEVICE),ci_b.to(DEVICE),
                ca_b.to(DEVICE), us_b.to(DEVICE),bz_b.to(DEVICE),y_b.to(DEVICE))
            preds = model(num_b,st_b,ci_b,ca_b,us_b,bz_b)
            loss  = criterion(preds, y_b)
            if optimizer:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            preds_all.extend(preds.detach().cpu().numpy())
            labels_all.extend(y_b.cpu().numpy())
    p=np.array(preds_all); l=np.array(labels_all)
    return (round(float(np.sqrt(mean_squared_error(l,p))),4),
            round(float(mean_absolute_error(l,p)),4))


In [ ]:
NUM_EPOCHS  = 30
PATIENCE    = 5
criterion   = nn.MSELoss()
optimizer   = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

best_val    = float('inf')
best_epoch  = 0
patience_cnt= 0
best_state  = None
history     = {'train_rmse':[], 'val_rmse':[]}

t0 = time.perf_counter()
for epoch in range(NUM_EPOCHS):
    tr_rmse, _  = run_epoch(model, train_loader, criterion, optimizer)
    val_rmse, _ = run_epoch(model, val_loader,   criterion)
    history['train_rmse'].append(tr_rmse)
    history['val_rmse'].append(val_rmse)

    if val_rmse < best_val:
        best_val   = val_rmse; best_epoch = epoch+1
        patience_cnt = 0
        best_state = {k:v.clone() for k,v in model.state_dict().items()}
    else:
        patience_cnt += 1

    if (epoch+1)%5==0 or epoch==0:
        print(f"Epoch {epoch+1:>2}/{NUM_EPOCHS}  "
              f"Train: {tr_rmse}  Val: {val_rmse}  "
              f"{'← best' if patience_cnt==0 else f'patience {patience_cnt}/{PATIENCE}'}")
    if patience_cnt >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

model.load_state_dict(best_state)
train_time = time.perf_counter() - t0
print(f"\nBest val RMSE: {best_val} at epoch {best_epoch}")
print(f"Training time: {train_time:.1f}s")


---
## Step 8 — Evaluate

In [ ]:
tr_rmse,  tr_mae  = run_epoch(model, train_loader, criterion)
val_rmse, val_mae = run_epoch(model, val_loader,   criterion)
te_rmse,  te_mae  = run_epoch(model, test_loader,  criterion)

print("=" * 55)
print("YelpEmbeddingANN — Final Results")
print("=" * 55)
print(f"  Train      RMSE: {tr_rmse}   MAE: {tr_mae}")
print(f"  Validation RMSE: {val_rmse}   MAE: {val_mae}")
print(f"  Test       RMSE: {te_rmse}   MAE: {te_mae}")
print(f"  Train time : {train_time:.1f}s")
print(f"  Best epoch : {best_epoch}")
print(f"  Train-Test gap : {round(te_rmse-tr_rmse,4)}")


---
## Step 9 — Save Embeddings to Disk

In [ ]:
# Extract all embedding weight matrices
weights = {
    'state_embeddings'   : model.state_emb.weight.detach().cpu().numpy(),
    'city_embeddings'    : model.city_emb.weight.detach().cpu().numpy(),
    'cat_embeddings'     : model.cat_emb.weight.detach().cpu().numpy(),
    'user_embeddings'    : model.user_emb.weight.detach().cpu().numpy(),
    'business_embeddings': model.biz_emb.weight.detach().cpu().numpy(),
}
for fname, w in weights.items():
    np.save(os.path.join(EMB_DIR, f'{fname}.npy'), w)
    print(f"  Saved {fname}.npy  {w.shape}")


In [ ]:
# Save ANN results
ann_results = {
    'model'       : 'YelpEmbeddingANN',
    'train_rmse'  : tr_rmse,  'val_rmse'  : val_rmse,  'test_rmse'  : te_rmse,
    'train_mae'   : tr_mae,   'val_mae'   : val_mae,   'test_mae'   : te_mae,
    'train_time_s': round(train_time,1), 'best_epoch': best_epoch
}
with open(os.path.join(EMB_DIR, 'ann_results.json'), 'w') as f:
    json.dump(ann_results, f, indent=2)

# Confirm all files
print("\nEmbedding folder:")
for fname in sorted(os.listdir(EMB_DIR)):
    size = os.path.getsize(os.path.join(EMB_DIR, fname)) / 1e3
    print(f"  {fname:<35}  {size:.1f} KB")
